# Pipeline d'entraînement MLlib — Scoring produits d'épargne (V1.1)

Consomme la sortie de `EDA_final.ipynb` (`PATH_TRAIN_OUT` / `PATH_SCORER_OUT` — parquet nettoyé,
catégorielles encore brutes). Encode, assemble, entraîne, évalue, sauvegarde, puis score.

**Principe de non-fuite** : tout ce qui est appris (indexeurs, encodeurs, poids de classe, modèle)
est **fit uniquement sur le train**, sauvegardé en `PipelineModel`, puis **rechargé tel quel** pour
scorer `dataset_a_scorer` — même logique que l'`Imputer`/les bornes IQR/les flags dans la Partie 1
EDA.

**Corrections apportées dans cette version (vs. V1) :**
- Ajout du **nettoyage de frontière par Tomek Links** (guide section 7.6bis), en pandas, sur le
  train étiqueté uniquement, **avant** le split train/validation.
- Ajout de la **pondération inverse-fréquence par classe** (guide section 7.6), recalculée
  **après** Tomek Links (la distribution des classes change), injectée via `weightCol` dans le
  classifieur.
- Correction d'une affirmation fausse en section 10 (V1) : `RandomForestClassifier` **supporte**
  `weightCol` depuis Spark 3.0 — le guide (section 7.5) l'utilise explicitement. La V1 prétendait
  le contraire, ce qui n'était pas cohérent avec le guide.
- Le lookup manuel `F.udf(...)` en section scoring remplacé par `IndexToString` (guide section
  7.3), qui est l'outil MLlib documenté pour cette conversion indice → nom de produit.

**Plan :**
1. Imports & configuration
2. Chargement du train nettoyé
3. Nettoyage de frontière (Tomek Links)
4. Split train / validation
5. Pondération inverse-fréquence par classe
6. Construction du pipeline (encodage → assemblage → modèle pondéré)
7. Entraînement
8. Évaluation (accuracy, F1 pondéré, matrice de confusion)
9. Importance des variables
10. Refit sur 100% du train (poids recalculés) & sauvegarde
11. Scoring (`dataset_a_scorer`)
12. Limites de cette V1.1 & prochaines étapes

## 1. Imports & configuration

In [ ]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, IndexToString
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# pip install imbalanced-learn --break-system-packages (si pas déjà présent dans l'environnement)
from imblearn.under_sampling import TomekLinks


In [ ]:
# ═══ Bascule LOCAL / MinIO — même principe que la Partie 1 EDA ═══
LOCAL_MODE = True

if LOCAL_MODE:
    PATH_TRAIN_IN = "new_test/part-00000_final.parquet"   # sortie de traiter_dataset(is_train=True)
    PATH_SCORER_IN = None                                    # pas encore testé en local
    PATH_PREDICTIONS_OUT = "new_test/predictions.parquet"
    MODEL_PATH = "./models/scoring_pipeline"
else:
    PATH_TRAIN_IN = "s3a://processed-data/dataset_train_produits_final/"
    PATH_SCORER_IN = "s3a://processed-data/dataset_a_scorer_final/"
    PATH_PREDICTIONS_OUT = "s3a://processed-data/predictions/"
    MODEL_PATH = "s3a://ml-scoring/models/scoring_pipeline"

RANDOM_SEED = 42


In [ ]:
# CORRECTIF POTENTIEL : ces deux listes DOIVENT rester identiques à celles de la
# Partie 1 EDA (section 0) -- si l'une évolue sans l'autre, le pipeline plante au
# fit (colonne absente) ou encode silencieusement moins de colonnes que prévu.
# TODO V2 : factoriser dans un module partagé (ex. config.py) importé par les deux
# notebooks, plutôt que dupliqué ici.
COLS_CATEGORIELLES_BASSE_CARDINALITE = [
    "GENDER", "TAILLE_ENTREPRI", "pack_actuel", "pack_etat",
    "CUSTOMER_RATING", "MARITAL_STATUS", "BPR",
]
COL_HAUTE_CARDINALITE = "CODE_VILLE"  # indexé seul, jamais one-hot (273 modalités)
COL_LABEL = "label_nom"
COLS_A_EXCLURE_DES_FEATURES = ["label_code", "label_nom"]  # cible, sous ses deux formes


## 2. Spark session & chargement du train nettoyé

In [ ]:
def get_spark() -> SparkSession:
    builder = SparkSession.builder.appName("training_pipeline_scoring")
    if LOCAL_MODE:
        builder = builder.master("local[*]")
    else:
        builder = (
            builder.master("spark://spark-master:7077")
            .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
            .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
            .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
            .config("spark.hadoop.fs.s3a.path.style.access", "true")
            .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        )
    spark = builder.getOrCreate()
    spark.sparkContext.setLogLevel("WARN")
    return spark


spark = get_spark()
df_train_full = spark.read.parquet(PATH_TRAIN_IN)
print(f"{df_train_full.count()} lignes, {len(df_train_full.columns)} colonnes")
df_train_full.printSchema()


In [ ]:
def colonnes_features_numeriques(df: DataFrame) -> list:
    """Toutes les colonnes numériques du dataset, hors cible et hors colonnes
    catégorielles couvertes par l'encodage (section 6) -- sinon on se retrouverait
    avec CODE_VILLE brut ET CODE_VILLE_idx dans le vecteur final. Utilisée à la
    fois pour Tomek Links (section 3, sur les features numériques uniquement --
    TomekLinks calcule des distances, incompatible avec des catégorielles brutes
    non encodées à ce stade) et pour VectorAssembler (section 6)."""
    cols_categorielles_brutes = set(COLS_CATEGORIELLES_BASSE_CARDINALITE + [COL_HAUTE_CARDINALITE])
    return [
        c for c, t in df.dtypes
        if t in ("int", "bigint", "double", "float")
        and c not in COLS_A_EXCLURE_DES_FEATURES
        and c not in cols_categorielles_brutes
    ]


feature_cols_numeriques = colonnes_features_numeriques(df_train_full)
print(f"Features numériques ({len(feature_cols_numeriques)}) : {feature_cols_numeriques}")


## 3. Nettoyage de frontière (Tomek Links)

Guide, section 7.6bis. Un lien de Tomek est une paire de points de classes différentes qui sont
chacun le plus proche voisin de l'autre — un signal de frontière, voire d'empiètement de la
classe majoritaire sur la zone d'une classe minoritaire. Les supprimer nettoie la frontière de
décision. **Complémentaire à la pondération de classe (section 5), pas un substitut** : Tomek
Links agit sur les *lignes* avant l'entraînement, la pondération agit sur la *fonction de coût*
pendant l'entraînement.

**Pourquoi en pandas et pas en Spark** : `imbalanced-learn` n'a pas d'équivalent distribué
natif dans MLlib. Ce n'est pas un problème ici : Tomek Links ne s'applique **qu'au train
étiqueté** (ce fichier), jamais à `dataset_a_scorer` (potentiellement 3M+ lignes une fois sur
MinIO) — un aller-retour pandas sur un fichier de cette taille est sans risque mémoire.

**Ordre à respecter** (guide) : Tomek Links tourne **avant** le split train/validation (section 4)
et **avant** le calcul des poids de classe (section 5, qui doit refléter la distribution
post-Tomek, pas la distribution brute).

In [ ]:
RADICAL_PRESENT = "RADICAL" in df_train_full.columns
cols_tomek = (["RADICAL"] if RADICAL_PRESENT else []) + feature_cols_numeriques + [COL_LABEL]

train_pd = df_train_full.select(*cols_tomek).toPandas()
print(f"Avant Tomek Links : {len(train_pd)} lignes")
print(train_pd[COL_LABEL].value_counts())

X = train_pd[feature_cols_numeriques]
y = train_pd[COL_LABEL]

tomek = TomekLinks(sampling_strategy="auto")
X_res, y_res = tomek.fit_resample(X, y)

# fit_resample renvoie de nouveaux index -- on les utilise pour retrouver les
# lignes conservées (RADICAL + toutes les colonnes d'origine), pas seulement
# les features numériques utilisées pour le calcul des distances.
indices_conserves = X_res.index
train_pd_clean = train_pd.loc[indices_conserves].reset_index(drop=True)

print(f"Après Tomek Links : {len(train_pd_clean)} lignes "
      f"({len(train_pd) - len(train_pd_clean)} supprimée(s), frontière entre classes)")
print(train_pd_clean[COL_LABEL].value_counts())

# Retour dans Spark : on rejoint train_pd_clean (juste RADICAL + label) à
# df_train_full pour récupérer TOUTES les colonnes (y compris les catégorielles
# brutes, absentes de train_pd puisque Tomek ne travaille que sur du numérique).
if RADICAL_PRESENT:
    radicaux_conserves = spark.createDataFrame(train_pd_clean[["RADICAL"]])
    df_train_full = df_train_full.join(radicaux_conserves, on="RADICAL", how="inner")
else:
    # Pas de RADICAL disponible (ex. fichier local de test déjà sans colonnes techniques) :
    # on retombe sur les lignes nettoyées telles quelles, sans les colonnes catégorielles
    # brutes -- acceptable uniquement si ces colonnes ne sont pas requises en aval.
    df_train_full = spark.createDataFrame(train_pd_clean)

df_train_full.cache()
print(f"\ndf_train_full après Tomek Links : {df_train_full.count()} lignes")


## 4. Split train / validation

Un split est fait ici **en plus** du train/scoring déjà séparé en amont (Partie 1 EDA) : celui-là
sépare "données qu'on a le droit d'utiliser" de "données de production", celui-ci sépare "données
pour entraîner" de "données pour évaluer honnêtement avant de livrer le modèle". Sans ce split, la
métrique d'évaluation serait mesurée sur les données mêmes qui ont servi à fitter les indexeurs et
le classifieur -- optimiste, pas fiable.

Split fait **après** Tomek Links (section 3), sur `df_train_full` déjà nettoyé -- le `PipelineModel`
final (section 10) sera de toute façon refit sur 100% de ce `df_train_full` nettoyé une fois la
V1.1 validée ici.

In [ ]:
df_fit, df_val = df_train_full.randomSplit([0.8, 0.2], seed=RANDOM_SEED)
df_fit.cache()
df_val.cache()
print(f"Fit  : {df_fit.count()} lignes")
print(f"Val  : {df_val.count()} lignes")

print("\nÉquilibre des classes -- fit vs val (doivent être comparables) :")
df_fit.groupBy(COL_LABEL).count().withColumn("part_fit", F.round(F.col("count") / df_fit.count(), 3)).show()
df_val.groupBy(COL_LABEL).count().withColumn("part_val", F.round(F.col("count") / df_val.count(), 3)).show()


## 5. Pondération inverse-fréquence par classe

Guide, section 7.6. Complémentaire à Tomek Links (section 3) : Tomek nettoie les lignes
ambiguës à la frontière, la pondération force le classifieur à ne pas ignorer les classes rares
pendant l'apprentissage. **Calculée après Tomek Links**, sur `df_fit` -- la distribution des
classes a changé suite au nettoyage de frontière, un poids calculé sur la distribution brute
d'avant-Tomek serait faux.

Formule (guide) : `poids_classe = total / (nb_classes × effectif_de_la_classe)` -- pondération
inverse-fréquence standard multi-classe, chaque classe rare reçoit un poids proportionnellement
plus élevé.

In [ ]:
def calculer_poids_classe(df: DataFrame, col_label: str = COL_LABEL) -> DataFrame:
    """Retourne un DataFrame [col_label, poids_classe] à joindre sur le train
    (fit ou full) juste avant l'entraînement. Recalculé à chaque fois que la
    population d'entrée change (ex. fit vs. full au refit de la section 10)."""
    effectifs = df.groupBy(col_label).count()
    total = df.count()
    nb_classes = effectifs.count()

    poids = effectifs.withColumn(
        "poids_classe", total / (nb_classes * F.col("count"))
    ).select(col_label, "poids_classe")

    print(f"\nPoids par classe (total={total}, nb_classes={nb_classes}) :")
    poids.orderBy(col_label).show()
    return poids


poids_par_classe_fit = calculer_poids_classe(df_fit)
df_fit = df_fit.join(poids_par_classe_fit, on=COL_LABEL)


## 6. Construction du pipeline

Trois familles de stages, dans l'ordre où Spark doit les exécuter :

1. **Encodage catégoriel** : `StringIndexer` + `OneHotEncoder` pour les colonnes basse-cardinalité,
   `StringIndexer` seul pour `CODE_VILLE` (haute cardinalité — un `OneHotEncoder` dessus
   exploserait le nombre de features pour un gain incertain).
2. **Indexation de la cible** : `label_nom` (string) → `label_idx` (double), requis par les
   classifieurs MLlib.
3. **Assemblage** : toutes les features numériques + toutes les sorties d'encodage dans un seul
   vecteur `features`, puis le classifieur, **pondéré par `poids_classe`** (section 5).

`RandomForestClassifier` choisi pour cette V1.1 : robuste aux features non standardisées, gère
nativement `CODE_VILLE_idx` à haute cardinalité sans que l'ordre numérique arbitraire de l'index
ne biaise le modèle (contrairement à une régression logistique), **et supporte `weightCol`
nativement depuis Spark 3.0** — exactement l'exemple du guide (section 7.5).

In [ ]:
def construire_stages_encodage(cols_basse_cardinalite: list, col_haute_cardinalite: str):
    indexers = [
        StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
        for c in cols_basse_cardinalite
    ]
    encoders = [
        OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
        for c in cols_basse_cardinalite
    ]
    indexer_haute_card = StringIndexer(
        inputCol=col_haute_cardinalite, outputCol=f"{col_haute_cardinalite}_idx", handleInvalid="keep"
    )
    return indexers + encoders + [indexer_haute_card]


encodage_stages = construire_stages_encodage(COLS_CATEGORIELLES_BASSE_CARDINALITE, COL_HAUTE_CARDINALITE)

label_indexer = StringIndexer(inputCol=COL_LABEL, outputCol="label_idx", handleInvalid="error")

feature_cols_encodees = [f"{c}_ohe" for c in COLS_CATEGORIELLES_BASSE_CARDINALITE] + [f"{COL_HAUTE_CARDINALITE}_idx"]
feature_cols = feature_cols_numeriques + feature_cols_encodees

print(f"Features numériques ({len(feature_cols_numeriques)}) : {feature_cols_numeriques}")
print(f"Features encodées   ({len(feature_cols_encodees)}) : {feature_cols_encodees}")

# handleInvalid="skip" : "poids_classe" n'est PAS dans inputCols, VectorAssembler
# ne le touche pas -- il reste disponible comme colonne à part pour weightCol ci-dessous.
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")

clf = RandomForestClassifier(
    labelCol="label_idx",
    featuresCol="features",
    predictionCol="prediction",
    probabilityCol="probability",
    weightCol="poids_classe",   # CORRECTIF : la V1 n'utilisait aucune pondération
    numTrees=100,
    maxDepth=8,
    seed=RANDOM_SEED,
)

pipeline = Pipeline(stages=encodage_stages + [label_indexer, assembler, clf])


## 7. Entraînement

Un seul `.fit()` déclenche, dans l'ordre, le fit de chaque `StringIndexer`/`OneHotEncoder` (appris
sur `df_fit` uniquement) puis l'entraînement du `RandomForestClassifier`, pondéré par
`poids_classe`.

In [ ]:
pipeline_model = pipeline.fit(df_fit)
print("Entraînement terminé.")


## 8. Évaluation sur le set de validation

Accuracy seule est trompeuse ici (cible déséquilibrée, cf. EDA Partie 2 section 3) — F1 pondéré et
la matrice de confusion donnent une image plus honnête, en particulier sur les classes
minoritaires. `df_val` n'a **pas** de colonne `poids_classe` (jointe uniquement sur `df_fit` en
section 5) — c'est volontaire, l'évaluation doit rester sur la distribution réelle, pas pondérée.

In [ ]:
predictions_val = pipeline_model.transform(df_val)

evaluator_acc = MulticlassClassificationEvaluator(labelCol="label_idx", predictionCol="prediction", metricName="accuracy")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label_idx", predictionCol="prediction", metricName="f1")
evaluator_precision = MulticlassClassificationEvaluator(labelCol="label_idx", predictionCol="prediction", metricName="weightedPrecision")
evaluator_recall = MulticlassClassificationEvaluator(labelCol="label_idx", predictionCol="prediction", metricName="weightedRecall")

print(f"Accuracy           : {evaluator_acc.evaluate(predictions_val):.4f}")
print(f"F1 pondéré         : {evaluator_f1.evaluate(predictions_val):.4f}")
print(f"Précision pondérée : {evaluator_precision.evaluate(predictions_val):.4f}")
print(f"Rappel pondéré     : {evaluator_recall.evaluate(predictions_val):.4f}")


In [ ]:
# Matrice de confusion (le set de validation est petit -- OK de collecter en pandas ;
# sur un dataset_a_scorer complet de plusieurs millions de lignes, échantillonner d'abord)
label_map = dict(enumerate(pipeline_model.stages[len(encodage_stages)].labels))  # label_idx -> label_nom
print("Correspondance label_idx -> label_nom :", label_map)

pred_pd = predictions_val.select("label_idx", "prediction").toPandas()
pred_pd["label_nom_reel"] = pred_pd["label_idx"].map(label_map)
pred_pd["label_nom_predit"] = pred_pd["prediction"].map(label_map)

confusion = pd.crosstab(pred_pd["label_nom_reel"], pred_pd["label_nom_predit"], margins=True)
print("\nMatrice de confusion (lignes = réel, colonnes = prédit) :")
print(confusion)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(pd.crosstab(pred_pd["label_nom_reel"], pred_pd["label_nom_predit"]), annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title("Matrice de confusion — validation")
plt.tight_layout()
plt.show()


## 9. Importance des variables

Utile pour vérifier que le modèle s'appuie sur des signaux plausibles (pas uniquement
`CODE_VILLE_idx` ou un artefact du feature engineering) avant de livrer la V1.1.

In [ ]:
rf_model = pipeline_model.stages[-1]  # dernier stage du pipeline = le RandomForestClassifier
importances = pd.Series(rf_model.featureImportances.toArray(), index=feature_cols)
importances = importances.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, max(4, 0.3 * len(importances))))
sns.barplot(x=importances.values, y=importances.index, ax=ax, color="#2980b9")
ax.set_xlabel("Importance (Gini)")
ax.set_title("Importance des variables — RandomForest")
plt.tight_layout()
plt.show()

importances.to_frame("importance")


## 10. Refit sur 100% du train (poids recalculés) & sauvegarde

Les métriques de la section 8 valident l'approche sur le split 80/20. Une fois satisfait des
résultats, le modèle **livré** est refit sur `df_train_full` (déjà nettoyé par Tomek Links, section
3) en entier — le split de la section 4 n'existe que pour évaluer honnêtement, il ne doit pas
priver le modèle final de 20% des données. `poids_classe` est **recalculé** ici sur
`df_train_full` : les poids appris sur `df_fit` (80%) ne sont pas exactement ceux de la population
complète.

In [ ]:
poids_par_classe_full = calculer_poids_classe(df_train_full)
df_train_full_pondere = df_train_full.join(poids_par_classe_full, on=COL_LABEL)

pipeline_model_final = pipeline.fit(df_train_full_pondere)

pipeline_model_final.write().overwrite().save(MODEL_PATH)
print(f"PipelineModel sauvegardé : {MODEL_PATH}")


## 11. Scoring (`dataset_a_scorer`)

Chargement du `PipelineModel` sauvegardé et `.transform()` — aucun `.fit()` ici, tout ce qui a été
appris (encodeurs, poids de classe implicites dans l'arbre entraîné, modèle) vient exclusivement
de `df_train_full`.

**CORRECTIF** : la V1 reconstruisait la correspondance indice → nom de produit à la main avec un
`F.udf(...)`. Le guide documente `IndexToString` (section 7.3) pour exactement ce besoin — utilisé
ici à la place.

In [ ]:
if PATH_SCORER_IN is not None:
    pipeline_model_reload = PipelineModel.load(MODEL_PATH)
    df_scorer = spark.read.parquet(PATH_SCORER_IN)

    predictions = pipeline_model_reload.transform(df_scorer)

    # Les labels appris par le label_indexer fitté restent accessibles tels quels
    # dans le PipelineModel rechargé -- pas besoin de refitter un indexeur à part
    # juste pour récupérer la liste, contrairement à l'exemple du guide qui refait
    # un indexer.fit(df_final).labels séparé.
    label_indexer_model = pipeline_model_reload.stages[len(encodage_stages)]
    converter = IndexToString(inputCol="prediction", outputCol="label_predit", labels=label_indexer_model.labels)
    predictions = converter.transform(predictions)

    cols_a_garder = [c for c in ["RADICAL", "CODE_VILLE"] if c in predictions.columns] + ["label_predit", "probability"]
    predictions.select(*cols_a_garder).show(20, truncate=False)

    predictions.write.mode("overwrite").parquet(PATH_PREDICTIONS_OUT)
    print(f"Prédictions écrites : {PATH_PREDICTIONS_OUT}")
else:
    print("PATH_SCORER_IN non défini (LOCAL_MODE=True, dataset_a_scorer pas encore testé en local) "
          "-- basculer LOCAL_MODE=False une fois prêt pour le cluster complet.")


## 12. Limites de cette V1.1 & prochaines étapes

Cette version corrige les deux lacunes signalées après la V1 (pas de pondération de classe, pas de
Tomek Links) et une affirmation fausse (voir ci-dessous). Points volontairement encore laissés de
côté, à traiter dans une V2 :

- **Correctif V1 → V1.1** : la V1 affirmait que `RandomForestClassifier` "ne supporte pas
  `weightCol` comme `LogisticRegression`". C'était faux et contredisait directement le guide
  (section 7.5, qui utilise `RandomForestClassifier(..., weightCol="poids_classe")`) — Spark 3.0+
  supporte `weightCol` sur `RandomForestClassifier`. Corrigé : la pondération est maintenant
  utilisée (section 5/6).
- **Pas de tuning d'hyperparamètres.** `numTrees=100, maxDepth=8` sont des valeurs de départ
  raisonnables, pas optimisées. À faire via `CrossValidator` + `ParamGridBuilder` sur `df_fit`
  (jamais sur `df_val`, qui doit rester un set d'évaluation propre — ou passer à un vrai
  train/val/test à 3 blocs si le tuning consomme le set de validation actuel).
- **Tomek Links non paramétré finement.** `sampling_strategy="auto"` nettoie toutes les classes
  majoritaires par défaut — à comparer avec une stratégie ciblée sur la classe la plus
  fréquente uniquement (`sampling_strategy={classe_majoritaire: ...}`) si le F1 des classes
  minoritaires ne s'améliore pas suffisamment.
- **Un seul algorithme essayé.** `RandomForestClassifier` choisi par défaut pour sa robustesse
  aux features non standardisées et à `CODE_VILLE_idx`, et son support natif de `weightCol`. À
  comparer avec `LogisticRegression` (`family="multinomial", weightCol=...`, nécessiterait un
  `StandardScaler` avant le `VectorAssembler`, cf. note EDA sur l'échelle des variables),
  `DecisionTreeClassifier` et `NaiveBayes` (ne supporte pas `weightCol` — guide section 7.9) :
  le guide donne la boucle de comparaison complète (section 7.9).
- **Pas de cross-validation.** Un split 80/20 unique donne une estimation, pas une borne de
  confiance. `CrossValidator` (k=5 par ex.) donnerait une évaluation plus robuste avant la
  décision finale de modèle.
- **`CODE_VILLE_idx` reste un entier arbitraire** pour un modèle à base d'arbres, ce qui est
  acceptable (RandomForest peut apprendre des seuils sur un index sans supposer d'ordre
  linéaire), mais deviendrait un problème si un modèle linéaire est testé en V2 — prévoir un
  encodage par fréquence ou par cible (`target encoding`) à ce moment-là plutôt qu'un `OneHotEncoder`
  brut (273 colonnes).
- **`MODEL_PATH` codé pour un seul modèle**, pas de versionnement — à ajouter (timestamp ou
  numéro de version dans le chemin) avant tout déploiement réel.
- **Tomek Links + refit final (section 10)** : le fichier local de test n'a probablement pas de
  colonne `RADICAL` si `LOCAL_MODE=True` pointe vers un parquet déjà réduit — dans ce cas, la
  branche de repli en section 3 (`RADICAL_PRESENT = False`) reconstruit `df_train_full`
  uniquement à partir des colonnes numériques utilisées par Tomek, ce qui **supprimerait les
  colonnes catégorielles brutes nécessaires à l'encodage (section 6)**. À vérifier explicitement
  avant tout run complet : si `RADICAL` n'est pas dans le parquet nettoyé, l'ajouter en amont
  (Partie 1 EDA) plutôt que de contourner ici.